# Notebook 1: Preprocesamiento de Datos
## Proyecto: Predicción de Churn en Telecomunicaciones

**Objetivo:** Explorar, limpiar y preparar el dataset *Telco Customer Churn* para el entrenamiento de modelos de Machine Learning.

**Dataset:** IBM Telco Customer Churn  
**Filas:** 7,043 clientes | **Columnas:** 21 variables  
**Variable objetivo:** `Churn` (¿El cliente abandonó el servicio?)

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print('Librerías importadas correctamente')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## 2. Cargar datos

In [ ]:
DATA_PATH = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 3. Análisis Exploratorio (EDA)

In [ ]:
# Información general del dataset
print('=== INFORMACIÓN DEL DATASET ===')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print(f'Memoria usada: {df.memory_usage(deep=True).sum() / 1024:.1f} KB\n')

print('=== TIPOS DE DATOS ===')
print(df.dtypes.value_counts())

print('\n=== VALORES NULOS ===')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No hay valores nulos explícitos')

In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe()

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de Churn (conteo)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Cantidad de clientes')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporción de Churn', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../artifacts/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nDesbalance de clases: {churn_counts["No"]}/{churn_counts["Yes"]} (No/Yes)')

In [ ]:
# Análisis de variables numéricas continuas
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(numeric_cols):
    df_temp = df[df[col].apply(lambda x: str(x).replace('.','',1).isdigit())].copy()
    df_temp[col] = pd.to_numeric(df_temp[col])
    
    axes[0, i].hist(df_temp[col], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'Distribución de {col}', fontweight='bold')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Frecuencia')
    
    no_churn = df_temp[df_temp['Churn'] == 'No'][col]
    yes_churn = df_temp[df_temp['Churn'] == 'Yes'][col]
    axes[1, i].hist(no_churn, bins=25, alpha=0.7, label='No Churn', color='#2ecc71')
    axes[1, i].hist(yes_churn, bins=25, alpha=0.7, label='Churn', color='#e74c3c')
    axes[1, i].set_title(f'{col} por Churn', fontweight='bold')
    axes[1, i].legend()

plt.suptitle('Variables Numéricas: Distribución y Relación con Churn', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../artifacts/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis de variables categóricas clave
cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'TechSupport', 'OnlineSecurity']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(20, 5))

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    churn_rate.sort_values(ascending=False).plot(
        kind='bar', ax=axes[i], color='#e74c3c', alpha=0.8, edgecolor='white'
    )
    axes[i].set_title(f'Tasa de Churn\npor {col}', fontweight='bold', fontsize=10)
    axes[i].set_ylabel('% Churn')
    axes[i].tick_params(axis='x', rotation=30)
    for p in axes[i].patches:
        axes[i].annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height()),
                        ha='center', va='bottom', fontsize=8)

plt.suptitle('Tasa de Churn por Variables Categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../artifacts/categorical_churn_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Limpieza de datos

In [ ]:
df_clean = df.copy()

# Eliminar columna ID (no aporta al modelo)
df_clean.drop(columns=['customerID'], inplace=True)
print(f'Columna customerID eliminada. Shape: {df_clean.shape}')

# TotalCharges tiene valores vacíos registrados como string ' '
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
missing_tc = df_clean['TotalCharges'].isnull().sum()
print(f'Valores nulos en TotalCharges después de conversión: {missing_tc}')

# Eliminar filas con valores nulos
df_clean.dropna(inplace=True)
print(f'Shape final después de dropna: {df_clean.shape}')
print(f'Filas eliminadas: {df.shape[0] - df_clean.shape[0]}')

## 5. Codificación de variables categóricas

In [ ]:
df_encoded = df_clean.copy()
label_encoders = {}

categorical_cols = df_encoded.select_dtypes(include=['object']).columns.tolist()
print(f'Variables categóricas a codificar ({len(categorical_cols)}): {categorical_cols}\n')

for col in categorical_cols:
    le = LabelEncoder()
    original_vals = df_encoded[col].unique().tolist()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f'{col}: {original_vals} → {le.classes_.tolist()}')

print(f'\nShape final: {df_encoded.shape}')
df_encoded.head(3)

## 6. Matriz de correlación

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))

corr_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    annot_kws={'size': 7}, ax=ax
)
ax.set_title('Matriz de Correlación - Telco Churn Dataset', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../artifacts/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlaciones con Churn
churn_corr = corr_matrix['Churn'].abs().sort_values(ascending=False).drop('Churn')
print('\nTop 10 variables más correlacionadas con Churn:')
print(churn_corr.head(10).to_string())

## 7. División y escalado del dataset

In [ ]:
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'Distribución del target: {y.value_counts().to_dict()}')
print(f'Features: {X.columns.tolist()}')

In [ ]:
# División estratificada para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Prueba:        {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nDistribución train - No Churn: {(y_train==0).sum()} | Churn: {(y_train==1).sum()}')
print(f'Distribución test  - No Churn: {(y_test==0).sum()} | Churn: {(y_test==1).sum()}')

In [ ]:
# Escalar features numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Escalado completado.')
print(f'Media de train (primeras 3 features): {X_train_scaled[:, :3].mean(axis=0).round(4)}')
print(f'Std de train  (primeras 3 features): {X_train_scaled[:, :3].std(axis=0).round(4)}')

## 8. Guardar artefactos

In [ ]:
# Guardar datos procesados
joblib.dump((X_train_scaled, X_test_scaled, y_train, y_test), '../artifacts/processed_data.pkl')
joblib.dump(scaler, '../artifacts/scaler.pkl')
joblib.dump(label_encoders, '../artifacts/label_encoders.pkl')
joblib.dump(X.columns.tolist(), '../artifacts/feature_names.pkl')

# Guardar dataset limpio como CSV para referencia
df_encoded.to_csv('../artifacts/churn_clean_encoded.csv', index=False)

print('Artefactos guardados en /artifacts/')
print('  - processed_data.pkl (X_train, X_test, y_train, y_test)')
print('  - scaler.pkl')
print('  - label_encoders.pkl')
print('  - feature_names.pkl')
print('  - churn_clean_encoded.csv')

## 9. Resumen del preprocesamiento

| Paso | Acción | Resultado |
|------|--------|-----------|
| Carga | Leer CSV | 7,043 filas × 21 columnas |
| Limpieza | Eliminar `customerID`, corregir `TotalCharges` | 7,032 filas × 20 columnas |
| Codificación | LabelEncoder en 16 variables categóricas | Todas numéricas |
| División | 80% train / 20% test (estratificado) | Train: 5,625 / Test: 1,407 |
| Escalado | StandardScaler | Media≈0, Std≈1 |

**Observaciones clave del EDA:**
- El dataset está **desbalanceado**: ~73% No Churn vs ~27% Churn
- Clientes con **contratos mes a mes** tienen la mayor tasa de churn (~43%)
- Clientes con **menos antigüedad (tenure)** tienen mayor tendencia a irse
- La **falta de seguridad online** se correlaciona fuertemente con el churn
- `TotalCharges` tiene alta correlación con `tenure` (multicolinealidad esperada)